# PRACTICA 1 #
## Procesado de datos masivo en Hadoop ##
Esta practica esta compuesta de dos ejercicios. El primero consiste en realizar operaciones de MapReduce con Python y el segundo en usar la herramiento Hive para tratar grandes cantidades de datos.
Al final de este documento, se incluira un apartado donde dare mis conclusiones y explicare como he usado la IA generativa para realizar esta tarea.

---

# EJERCICIO 1: MapReduce #
Primero cargaremos los datos de los ficheros usuarios.csv y eventos.csv en pandas para poder explorarlos:

In [99]:
import pandas as pd

columnas_usuarios = ['usuario_id', 'edad', 'sexo', 'país', 'titulación', 'fecha_alta']
columnas_eventos = ['evento_id', 'usuario_id', 'fecha_hora', 'curso', 'tipo_evento', 'recurso', 'duracion_segundos', 'resultado']

df_usuarios = pd.read_csv("usuarios.csv", names=columnas_usuarios, sep=';')
df_eventos = pd.read_csv("eventos.csv", names=columnas_eventos, sep=';')

In [100]:
df_usuarios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   usuario_id  16 non-null     object
 1   edad        16 non-null     object
 2   sexo        16 non-null     object
 3   país        16 non-null     object
 4   titulación  16 non-null     object
 5   fecha_alta  16 non-null     object
dtypes: object(6)
memory usage: 896.0+ bytes


In [101]:
df_eventos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   evento_id          68 non-null     object
 1   usuario_id         68 non-null     object
 2   fecha_hora         68 non-null     object
 3   curso              68 non-null     object
 4   tipo_evento        68 non-null     object
 5   recurso            68 non-null     object
 6   duracion_segundos  68 non-null     object
 7   resultado          22 non-null     object
dtypes: object(8)
memory usage: 4.4+ KB


In [102]:
df_eventos.head(3)

,evento_id,usuario_id,fecha_hora,curso,tipo_evento,recurso,duracion_segundos,resultado
0,evento_id,usuario_id,fecha_hora,curso,tipo_evento,recurso,duracion_segundos,resultado
1,E001,U001,2025-03-01T08:15:00,BigData,login,plataforma,120,NaN
2,E002,U001,2025-03-01T08:18:00,BigData,video,introduccion,840,NaN


In [103]:
df_usuarios.head(3)

,usuario_id,edad,sexo,país,titulación,fecha_alta
0,usuario_id,edad,sexo,pais,titulacion,fecha_alta
1,U001,24,F,España,Informática,2026-01-10
2,U002,31,M,España,Telecomunicación,2026-01-12


Como podemos observar, en el dataset se ha colado una fila con los nombres da cada una de las columnas. Tendriamos que eliminar esta fila.
También pondremos que el id de cada fila corresponda con `usuario_id` en la tabla de usuarios y con `evento_id` en la de eventos.

In [104]:
df_eventos[df_eventos['evento_id']=="evento_id"]

,evento_id,usuario_id,fecha_hora,curso,tipo_evento,recurso,duracion_segundos,resultado
0,evento_id,usuario_id,fecha_hora,curso,tipo_evento,recurso,duracion_segundos,resultado


In [105]:
df_eventos = df_eventos[df_eventos['evento_id'] != 'evento_id'].reset_index(drop=True)
df_usuarios = df_usuarios[df_usuarios['usuario_id'] != 'usuario_id'].reset_index(drop=True)

In [106]:
df_eventos.head(3)

,evento_id,usuario_id,fecha_hora,curso,tipo_evento,recurso,duracion_segundos,resultado
0,E001,U001,2025-03-01T08:15:00,BigData,login,plataforma,120,NaN
1,E002,U001,2025-03-01T08:18:00,BigData,video,introduccion,840,NaN
2,E003,U002,2025-03-01T09:05:00,BigData,login,plataforma,95,NaN


In [107]:
df_usuarios.head(3)

,usuario_id,edad,sexo,país,titulación,fecha_alta
0,U001,24,F,España,Informática,2026-01-10
1,U002,31,M,España,Telecomunicación,2026-01-12
2,U003,27,F,Portugal,Matemáticas,2026-01-15


In [108]:
df_eventos['tipo_evento'].value_counts()

tipo_evento
evaluacion    21
video         16
login         15
documento      8
foro           7
Name: count, dtype: int64

Ahora, subieremos los ficheros csv a hdfs y luego procederemos a realizar el código del reduce:

In [109]:
! hdfs dfs -mkdir /ficheros_ejercicio1
! hdfs dfs -put usuarios.csv /ficheros_ejercicio1
! hdfs dfs -put eventos.csv /ficheros_ejercicio1
! hdfs dfs -ls /ficheros_ejercicio1

mkdir: `/ficheros_ejercicio1': File exists
put: `/ficheros_ejercicio1/usuarios.csv': File exists
put: `/ficheros_ejercicio1/eventos.csv': File exists
Found 5 items
-rw-r--r--   3 root supergroup       4373 2026-11-11 10:39 /ficheros_ejercicio1/eventos.csv
drwxr-xr-x   - root supergroup          0 2026-11-11 11:35 /ficheros_ejercicio1/salida
drwxr-xr-x   - root supergroup          0 2026-11-11 13:02 /ficheros_ejercicio1/salida_ej2
drwxr-xr-x   - root supergroup          0 2026-11-11 13:01 /ficheros_ejercicio1/salida_ej2_join
-rw-r--r--   3 root supergroup        719 2026-11-11 10:39 /ficheros_ejercicio1/usuarios.csv


In [110]:
! mkdir ejercicio1

mkdir: cannot create directory ‘ejercicio1’: File exists


---
## EJERCICIO 1.1: Contador de eventos ##
Ahora, procederemos a realizar los ficheros python que se encargaran de realizar las funciones de mapreduce

In [111]:
%%writefile ejercicio1/mapper_ej1.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    splitted = linea.strip().split(';')
    if(splitted[0]=="evento_id"):
        continue
    tipo = splitted[4]
    print(f"{tipo}")

Overwriting ejercicio1/mapper_ej1.py


In [112]:
%%writefile ejercicio1/reducer_ej1.py
#!/usr/bin/env python3
import sys

result = {}

for linea in sys.stdin:
    tipo = linea.strip()
    if not tipo:
        continue
    result[tipo] = result.get(tipo,0) + 1 

for key,value in result.items():
    print(f"{key}\t{value}")
    

Overwriting ejercicio1/reducer_ej1.py


In [113]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files ejercicio1/mapper_ej1.py,ejercicio1/reducer_ej1.py \
-mapper "python3 mapper_ej1.py" \
-reducer "python3 reducer_ej1.py" \
-input /ficheros_ejercicio1/eventos.csv \
-output /ficheros_ejercicio1/salida/eventos_tipo.csv

Deleted /ficheros_ejercicio1/salida
packageJobJar: [/tmp/hadoop-unjar8952528489999850606/] [] /tmp/streamjob8291601066230910617.jar tmpDir=null
2026-11-11 13:13:01,307 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:13:01,487 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:13:01,782 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794390046936_0007
2026-11-11 13:13:02,375 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-11 13:13:02,473 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-11 13:13:02,610 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794390046936_0007
2026-11-11 13:13:02,611 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-11 13:13:02,814 INFO conf.Configuration: resource-types.xml not found
2026-11-11 13:1

In [114]:
! hdfs dfs -cat /ficheros_ejercicio1/salida/eventos_tipo.csv/part-*

documento	8
evaluacion	21
foro	7
login	15
video	16


In [115]:
df_eventos.groupby('tipo_evento').size()

tipo_evento
documento      8
evaluacion    21
foro           7
login         15
video         16
dtype: int64

El MapReduce funciona correctamente ya que devuelve el mismo resultado que si la operación se hubiera realizado usando pandas.

---
## EJERCICIO 1.2: Contador de evento por pais ##
Partiendo del ejercicio anterior, mejóralo para que devuelva el número de eventos realizado por usuarios de cada país. Para ello es necesario reunir ambos ficheros, ya que el fichero de eventos almacena el identificador de usuario (usuario_id), y el país se almacena solo en el fichero de usuarios.

In [116]:
%%writefile ejercicio1/mapper_ej2_join.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    campos = linea.strip().split(';')

    if campos[0] in ("usuario_id", "evento_id"):
        continue

    if len(campos) == 6:          # usuarios.csv
        usuario_id = campos[0]
        pais = campos[3].strip()
        print(f"{usuario_id}\tU\t{pais}")

    elif len(campos) == 8:        # eventos.csv
        usuario_id = campos[1]
        print(f"{usuario_id}\tE")

Overwriting ejercicio1/mapper_ej2_join.py


Tras la fase de mapper, el mecanismo shuffle & short ordena las salidas en función de la clave de usuario. De esta manera, los eventos y el pais del usuario apareceran de forma contigua y facilitaran el trabajo al siguiente mapper (el cual se encargara de hacer el join). 

In [117]:
%%writefile ejercicio1/reducer_ej2_join.py
#!/usr/bin/env python3
import sys

usuario_actual = None
pais = None
n_eventos = 0

def emitir(pais, n_eventos):
    if n_eventos > 0:
        print(f"{pais if pais else 'desconocido'}\t{n_eventos}")

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) < 2:
        continue
    usuario_id, etiqueta = campos[0], campos[1]

    # Cambio de usuario: emitir el anterior y reiniciar
    if usuario_id != usuario_actual:
        if usuario_actual is not None:
            emitir(pais, n_eventos)
        usuario_actual = usuario_id
        pais = None
        n_eventos = 0

    if etiqueta == "U":
        pais = campos[2]
    elif etiqueta == "E":
        n_eventos += 1

# último usuario
if usuario_actual is not None:
    emitir(pais, n_eventos)

Overwriting ejercicio1/reducer_ej2_join.py


El mapper anterior emitira pais y número de eventos por cada uno de los usuarios. El siguiente se encargara de sumar el total de eventos de todos los usuarios en función de su pais.

In [118]:
%%writefile ejercicio1/reducer_ej2_suma.py
#!/usr/bin/env python3
import sys

result = {}

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) != 2:
        continue
    pais, n = campos
    result[pais] = result.get(pais, 0) + int(n)

for pais, total in result.items():
    print(f"{pais}\t{total}")

Overwriting ejercicio1/reducer_ej2_suma.py


In [93]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej2_join
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/mapper_ej2_join.py,ejercicio1/reducer_ej2_join.py \
    -mapper "python3 mapper_ej2_join.py" \
    -reducer "python3 reducer_ej2_join.py" \
    -input /ficheros_ejercicio1/usuarios.csv \
    -input /ficheros_ejercicio1/eventos.csv \
    -output /ficheros_ejercicio1/salida_ej2_join

packageJobJar: [/tmp/hadoop-unjar220335860555716574/] [] /tmp/streamjob1534554475519735877.jar tmpDir=null
2026-11-11 13:01:22,255 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:01:22,433 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:01:22,763 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794390046936_0005
2026-11-11 13:01:23,403 INFO mapred.FileInputFormat: Total input files to process : 2
2026-11-11 13:01:23,502 INFO mapreduce.JobSubmitter: number of splits:3
2026-11-11 13:01:23,647 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794390046936_0005
2026-11-11 13:01:23,647 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-11 13:01:23,852 INFO conf.Configuration: resource-types.xml not found
2026-11-11 13:01:23,853 INFO resource.ResourceUtils:

In [94]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej2
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/reducer_ej2_suma.py \
    -mapper cat \
    -reducer "python3 reducer_ej2_suma.py" \
    -input /ficheros_ejercicio1/salida_ej2_join \
    -output /ficheros_ejercicio1/salida_ej2

packageJobJar: [/tmp/hadoop-unjar3353323633765511278/] [] /tmp/streamjob4313717906354216169.jar tmpDir=null
2026-11-11 13:01:53,782 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:01:53,984 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:01:54,276 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794390046936_0006
2026-11-11 13:01:54,794 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-11 13:01:54,876 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-11 13:01:55,023 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794390046936_0006
2026-11-11 13:01:55,023 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-11 13:01:55,235 INFO conf.Configuration: resource-types.xml not found
2026-11-11 13:01:55,236 INFO resource.ResourceUtils

In [95]:
! hdfs dfs -cat /ficheros_ejercicio1/salida_ej2/part-*

Argentina	4
Chile	4
Colombia	5
España	36
México	5
Perú	4
Portugal	9


In [119]:
resultado = (
    df_eventos
    .merge(df_usuarios[['usuario_id', 'país']], on='usuario_id', how='left')
    .groupby('país', dropna=False)
    .size()
    .reset_index(name='numero_eventos')
)
resultado

,país,numero_eventos
0,Argentina,4
1,Chile,4
2,Colombia,5
3,España,36
4,México,5
5,Perú,4
6,Portugal,9


Como podemos comprobar, funciona perfectamente.

---

## EJERCICIO 1.3: Mejorando el contador de eventos por país ##
Ahora, partiendo de los ejercicios anteriores, hay que hacer para que cada país y tipo de evento calcule la duración media de los eventos expresada en segundos.

In [127]:
%%writefile ejercicio1/mapper_ej3_join.py
#!/usr/bin/env python3
import sys

for linea in sys.stdin:
    campos = linea.strip().split(';')

    if campos[0] in ("usuario_id", "evento_id"):
        continue

    if len(campos) == 6:          # usuarios.csv
        usuario_id = campos[0]
        pais = campos[3].strip()
        print(f"{usuario_id}\tU\t{pais}")

    elif len(campos) == 8:        # eventos.csv
        usuario_id = campos[1]
        tipo = campos[4].strip()
        duracion = campos[6].strip()
        if duracion.isdigit():
            print(f"{usuario_id}\tE\t{tipo}\t{duracion}")

Overwriting ejercicio1/mapper_ej3_join.py


In [128]:
%%writefile ejercicio1/reducer_ej3_join.py
#!/usr/bin/env python3
import sys

usuario_actual = None
pais = None
por_tipo = {}          # {tipo: [suma_duracion, n_eventos]}

def emitir(pais, por_tipo):
    for tipo, (suma, n) in por_tipo.items():
        print(f"{pais if pais else 'desconocido'}\t{tipo}\t{suma}\t{n}")

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) < 2:
        continue
    usuario_id, etiqueta = campos[0], campos[1]

    if usuario_id != usuario_actual:
        if usuario_actual is not None:
            emitir(pais, por_tipo)
        usuario_actual = usuario_id
        pais = None
        por_tipo = {}

    if etiqueta == "U":
        pais = campos[2]
    elif etiqueta == "E" and len(campos) == 4:
        tipo, duracion = campos[2], int(campos[3])
        acum = por_tipo.setdefault(tipo, [0, 0])
        acum[0] += duracion
        acum[1] += 1

if usuario_actual is not None:
    emitir(pais, por_tipo)

Overwriting ejercicio1/reducer_ej3_join.py


In [129]:
%%writefile ejercicio1/reducer_ej3_media.py
#!/usr/bin/env python3
import sys

result = {}            # {(pais, tipo): [suma_duracion, n_eventos]}

for linea in sys.stdin:
    campos = linea.strip().split('\t')
    if len(campos) != 4:
        continue
    pais, tipo, suma, n = campos
    acum = result.setdefault((pais, tipo), [0, 0])
    acum[0] += int(suma)
    acum[1] += int(n)

for (pais, tipo), (suma, n) in sorted(result.items()):
    print(f"{pais}\t{tipo}\t{suma / n:.2f}")

Overwriting ejercicio1/reducer_ej3_media.py


In [130]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej3_join
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files ejercicio1/mapper_ej3_join.py,ejercicio1/reducer_ej3_join.py \
    -mapper "python3 mapper_ej3_join.py" \
    -reducer "python3 reducer_ej3_join.py" \
    -input /ficheros_ejercicio1/usuarios.csv \
    -input /ficheros_ejercicio1/eventos.csv \
    -output /ficheros_ejercicio1/salida_ej3_join

packageJobJar: [/tmp/hadoop-unjar4955396940111554330/] [] /tmp/streamjob1116835066777543903.jar tmpDir=null
2026-11-11 13:41:11,110 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:41:11,296 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:41:11,590 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794390046936_0009
2026-11-11 13:41:12,191 INFO mapred.FileInputFormat: Total input files to process : 2
2026-11-11 13:41:12,307 INFO mapreduce.JobSubmitter: number of splits:3
2026-11-11 13:41:12,465 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794390046936_0009
2026-11-11 13:41:12,465 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-11 13:41:12,698 INFO conf.Configuration: resource-types.xml not found
2026-11-11 13:41:12,698 INFO resource.ResourceUtils

In [131]:
! hdfs dfs -rm -r -f /ficheros_ejercicio1/salida_ej3
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -D stream.num.map.output.key.fields=2 \
    -files ejercicio1/reducer_ej3_media.py \
    -mapper cat \
    -reducer "python3 reducer_ej3_media.py" \
    -input /ficheros_ejercicio1/salida_ej3_join \
    -output /ficheros_ejercicio1/salida_ej3

packageJobJar: [/tmp/hadoop-unjar3804962196216786696/] [] /tmp/streamjob9072426275436993653.jar tmpDir=null
2026-11-11 13:43:33,561 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:43:33,755 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-11-11 13:43:34,031 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794390046936_0010
2026-11-11 13:43:34,511 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-11 13:43:34,603 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-11 13:43:34,740 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794390046936_0010
2026-11-11 13:43:34,741 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-11 13:43:34,946 INFO conf.Configuration: resource-types.xml not found
2026-11-11 13:43:34,947 INFO resource.ResourceUtils

In [132]:
! hdfs dfs -cat /ficheros_ejercicio1/salida_ej3/part-*

Argentina	documento	540.00
Argentina	evaluacion	910.00
Argentina	login	100.00
Argentina	video	900.00
Chile	documento	510.00
Chile	evaluacion	920.00
Chile	login	130.00
Chile	video	960.00
Colombia	evaluacion	895.00
Colombia	foro	720.00
Colombia	login	115.00
Colombia	video	840.00
España	documento	540.00
España	evaluacion	895.45
España	foro	532.50
España	login	93.75
España	video	903.33
México	evaluacion	920.00
México	foro	540.00
México	login	105.00
México	video	1200.00
Perú	evaluacion	930.00
Perú	foro	660.00
Perú	login	90.00
Perú	video	870.00
Portugal	documento	525.00
Portugal	evaluacion	913.33
Portugal	login	117.50
Portugal	video	975.00


In [134]:
df_join = df_eventos.merge(df_usuarios[['usuario_id', 'país']], on='usuario_id', how='left')

df_join['duracion_segundos'] = pd.to_numeric(df_join['duracion_segundos'], errors='coerce')

resultado = (
    df_join
    .groupby(['país', 'tipo_evento'], dropna=False)['duracion_segundos']
    .mean()
    .round(2)
    .reset_index(name='duracion_media')
)
resultado

,país,tipo_evento,duracion_media
0,Argentina,documento,540.00
1,Argentina,evaluacion,910.00
2,Argentina,login,100.00
3,Argentina,video,900.00
4,Chile,documento,510.00
5,Chile,evaluacion,920.00
6,Chile,login,130.00
7,Chile,video,960.00
8,Colombia,evaluacion,895.00
9,Colombia,foro,720.00
